# Product Review Sentiment Classification

This notebook trains and evaluates a reproducible text-classification pipeline for positive and negative product reviews.


In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
DATA_PATH = Path("product_sentiment.xlsx")
MODEL_PATH = Path("model-aziz-2.pkl")


## 1. Load and validate the data


In [ ]:
data = pd.read_excel(DATA_PATH)
required = {"review", "class"}
missing = required.difference(data.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
data = data.dropna(subset=["review", "class"]).drop_duplicates(subset=["review"]).copy()
data["review"] = data["review"].astype(str).str.strip()
data = data[data["review"].str.len() > 0]
print(data["class"].value_counts())
data.head()


In [ ]:
data["class"].value_counts().plot(kind="bar", title="Class distribution")
plt.xlabel("Sentiment class"); plt.ylabel("Reviews"); plt.tight_layout(); plt.show()


## 2. Build a reproducible TF-IDF classification pipeline


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["review"], data["class"], test_size=0.20, stratify=data["class"], random_state=SEED
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=SEED)),
])
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1_weighted")
print(f"Cross-validation F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
model.fit(X_train, y_train)


## 3. Evaluate on the held-out test set


In [ ]:
prediction = model.predict(X_test)
print(classification_report(y_test, prediction))
ConfusionMatrixDisplay.from_predictions(y_test, prediction, cmap="Blues")
plt.tight_layout(); plt.show()


## 4. Save the complete preprocessing-and-model pipeline


In [ ]:
joblib.dump(model, MODEL_PATH)
print(f"Saved model to {MODEL_PATH.resolve()}")


## Limitations

Sentiment labels and vocabulary are dataset-specific. Performance should be re-evaluated before using the model for a new product category, language or source platform.
